In [ ]:
import os
os.chdir("/content/drive/MyDrive/JCAIEAH-003/Notes and Hands On/Modul 2/Day 6/")

This cell changes the current working directory to access the data file.

In [ ]:
import pandas as pd

df = pd.read_json('data/News_Category_Dataset_v3.json', lines=True)
df.shape

(209527, 6)

This cell imports the pandas library and loads the news category dataset from a JSON file into a DataFrame, then displays its shape.

In [ ]:
df.head()

,link,headline,category,short_description,authors,date
0,https://www.huffpost.com/entry/covid-boosters-...,Over 4 Million Americans Roll Up Sleeves For O...,U.S. NEWS,Health experts said it is too early to predict...,"Carla K. Johnson, AP",2022-09-23
1,https://www.huffpost.com/entry/american-airlin...,"American Airlines Flyer Charged, Banned For Li...",U.S. NEWS,He was subdued by passengers and crew when he ...,Mary Papenfuss,2022-09-23
2,https://www.huffpost.com/entry/funniest-tweets...,23 Of The Funniest Tweets About Cats And Dogs ...,COMEDY,"""Until you have a dog you don't understand wha...",Elyse Wanshel,2022-09-23
3,https://www.huffpost.com/entry/funniest-parent...,The Funniest Tweets From Parents This Week (Se...,PARENTING,"""Accidentally put grown-up toothpaste on my to...",Caroline Bologna,2022-09-23
4,https://www.huffpost.com/entry/amy-cooper-lose...,Woman Who Called Cops On Black Bird-Watcher Lo...,U.S. NEWS,Amy Cooper accused investment firm Franklin Te...,Nina Golgowski,2022-09-22


This cell displays the first few rows of the DataFrame to inspect its structure and content.

In [ ]:
df['category'].value_counts()

,count
category,
POLITICS,35602
WELLNESS,17945
ENTERTAINMENT,17362
TRAVEL,9900
STYLE & BEAUTY,9814
PARENTING,8791
HEALTHY LIVING,6694
QUEER VOICES,6347
FOOD & DRINK,6340


This cell shows the distribution of news categories in the dataset.

In [ ]:
news = df[['short_description', 'category']]
news = news[news["category"].isin(["COMEDY", "BUSINESS", "SPORTS"])].reset_index(drop=True)
news = news.drop_duplicates(subset='short_description')
print(news.shape)
news.head()

(14131, 2)


,short_description,category
0,"""Until you have a dog you don't understand wha...",COMEDY
1,"Maury Wills, who helped the Los Angeles Dodger...",SPORTS
2,Las Vegas never had a professional sports cham...,SPORTS
3,The race's organizers say nonbinary athletes w...,SPORTS
4,Varvaro pitched mostly with the Atlanta Braves...,SPORTS


This cell preprocesses the data by selecting relevant columns, filtering for specific categories (COMEDY, BUSINESS, SPORTS), removing duplicate descriptions, and then displays the shape and head of the new 'news' DataFrame.

In [ ]:
news.isnull().sum()

,0
short_description,0
category,0


This cell checks for any missing values in the 'news' DataFrame.

In [ ]:
news.nunique()

,0
short_description,14131
category,3


This cell counts the number of unique values in each column of the 'news' DataFrame.

In [ ]:
news["category"].value_counts()

,count
category,
BUSINESS,5122
COMEDY,4619
SPORTS,4390


This cell displays the distribution of categories after filtering.

In [ ]:
import nltk, re
from nltk.tokenize import word_tokenize

from nltk.stem import PorterStemmer, SnowballStemmer
from nltk.stem import WordNetLemmatizer

nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

This cell imports necessary NLTK libraries for text preprocessing (tokenization, stemming, lemmatization) and downloads required NLTK data.

In [ ]:
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

This cell initializes the Porter Stemmer and WordNet Lemmatizer for text normalization.

In [ ]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import re

# Initialize stopwords set
stop_words = set(stopwords.words('english'))

# Define a helper function to remove stopwords
def remove_stopwords_from_text(text):
    tokens = word_tokenize(text)
    filtered_words = [word for word in tokens if word.lower() not in stop_words]
    return " ".join(filtered_words)

news["clean_description"] = news["short_description"].str.lower()
# Apply regex substitution to remove non-alphanumeric characters first
news["clean_description"] = news["clean_description"].apply(lambda x: re.sub(r"[^a-zA-Z0-9]", " ", x))
# Then remove stopwords
news["clean_description"] = news["clean_description"].apply(remove_stopwords_from_text)
# Then lemmatize
news["clean_description"] = news["clean_description"].apply(lambda x: " ".join([lemmatizer.lemmatize(word) for word in word_tokenize(x)]))
# Then stem
news["clean_description"] = news["clean_description"].apply(lambda x: " ".join([stemmer.stem(word) for word in word_tokenize(x)]))
news.head()

,short_description,category,clean_description
0,"""Until you have a dog you don't understand wha...",COMEDY,dog understand could eaten
1,"Maury Wills, who helped the Los Angeles Dodger...",SPORTS,mauri will help lo angel dodger win three worl...
2,Las Vegas never had a professional sports cham...,SPORTS,la vega never profession sport champion sunday
3,The race's organizers say nonbinary athletes w...,SPORTS,race organ say nonbinari athlet regist men wom...
4,Varvaro pitched mostly with the Atlanta Braves...,SPORTS,varvaro pitch mostli atlanta brave start law e...


This cell performs comprehensive text cleaning on the 'short_description' column, including converting to lowercase, removing non-alphanumeric characters, removing stopwords, lemmatization, and stemming, and stores the result in a new 'clean_description' column.

In [ ]:
X = news["clean_description"]
y = news["category"]

This cell separates the preprocessed text features (X) and the target category labels (y) for model training.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)
print(f"Data train: {X_train.shape}, {y_train.shape}")
print(f"Data test: {X_test.shape}, {y_test.shape}")

Data train: (11304,), (11304,)
Data test: (2827,), (2827,)


This cell splits the data into training and testing sets to evaluate the model's performance.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()

X_train_vec = tfidf.fit_transform(X_train)
X_test_vec = tfidf.transform(X_test)

This cell vectorizes the text data using TF-IDF, transforming the text into numerical features for the machine learning model.

In [ ]:
%%time
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier()
model.fit(X_train_vec, y_train)

CPU times: user 21 s, sys: 51.8 ms, total: 21 s
Wall time: 21.2 s


RandomForestClassifier()

This cell trains a RandomForestClassifier model on the TF-IDF vectorized training data and measures the training time.

In [ ]:
from sklearn.metrics import classification_report

y_pred = model.predict(X_test_vec)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

    BUSINESS       0.82      0.79      0.81      1023
      COMEDY       0.65      0.74      0.69       925
      SPORTS       0.74      0.66      0.69       879

    accuracy                           0.73      2827
   macro avg       0.74      0.73      0.73      2827
weighted avg       0.74      0.73      0.73      2827



This cell evaluates the trained model's performance on the test set and prints a classification report.